# E-DAXING — passive device provenance on modern smartphones

**Single self-contained notebook.** Mounts Drive, logs into Hugging Face, runs every stage.
Stages are resumable and each one halts on a failed gate rather than producing a number
you cannot trust.

---

## What the data actually supports

Read from the byte-verified inventory of *this* copy, not the upstream README.

**The iPhone 6s plan does not work.** This copy holds stills for only 40 of 85 devices, and
the entire Apple range contributes 246 images from a single device (2702). All ten iPhone 6s
bodies are video-only.

Same-model groups that do have stills:

| model | devices | images each | resolution |
|---|---|---|---|
| **Huawei EML-AL00** | **1101–1105 (5)** | 576–670 | 3968×2976 |
| OPPO R11 | 3301, 3302, 3304, 3305 | 423–489 | 4608×3456 |
| vivo X9 | 4101–4104 | 458–480 | 4608×3456 |
| Huawei EVA-AL10 | 1401, 1402, 1403, 1405 | 417–540 | 3968×2976 |

**EML-AL00 is the default**: five bodies of one model, mirroring the Kodak crossed design on
a 2018 smartphone. The others are one config change away.

## Three constraints this notebook is built around

**1 · Angle folders cannot be pooled blindly.** `90` is portrait; `0` and `180` are landscape
and differ by a 180° phone rotation. If the camera baked that rotation into pixels rather than
EXIF, PRNU alignment breaks between them. We use **`90` alone** by default, and D1 runs a
decisive test of whether `0` and `180` are actually compatible before you consider pooling.

**2 · Splits shrink.** The smallest `90` folder holds 244 images (device 1105) against
Dresden's 380, so `E2` drops from 140 to ~90. Fingerprint reliability will be lower and the
notebook reports it rather than hiding it.

**3 · No held-out same-model body exists.** All five EML-AL00 devices are in the design, so
$\kappa_{\mathrm{model}}$ comes from **leave-one-device-out**, exactly as in the Kodak study.

## The gate that matters
Modern smartphones apply multi-frame fusion and aggressive denoising. **D1 halts if PRNU
cannot separate these five bodies on held-out real images.** If that gate fails, the honest
result is "PRNU-based device identity is not recoverable from this device class at this crop",
which is itself publishable — do not tune around it.

## Stages
`D0` discover, split, scene audit → `D1` fingerprints, positive control, orientation test,
LODO κ → `D2` autoencoder round-trip retention → `D3` ten adapters → `D4` generation →
`D5` measurement in both representations → `D6` analysis. Roughly 11 GPU-hours after D0/D1.

In [ ]:
# %% 1 · mount, Hugging Face login, imports, configuration
import os, sys, json, glob, re, math, time, gc, shutil, hashlib, subprocess, random, itertools
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import csv as _csv

HF_TOKEN = ""            # optional: paste here, or store as a Colab secret named HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE = "/content/drive/MyDrive"
except Exception:
    DRIVE = os.environ.get("EINV_DRIVE", ".")

@dataclass
class CFG:
    STAGES: tuple = ("D0",)                    # D0 -> D1 -> D2 -> D3 -> D4 -> D5 -> D6
    DAXING: str = ""                           # blank = auto-locate under DRIVE
    OUT: str = "inv_channel/E_DAXING"
    MODEL_NAME: str = "Huawei P20 (EML-AL00)"               # for the record only
    DEVICES: tuple = ("1101","1102","1103","1104","1105")
    CROSS_MODEL: tuple = ("1201","3301","4101") # different-model nulls, one image each group
    ANGLE: str = "90"                          # ONE angle only; see D1's orientation test
    CROP: int = 1024
    N_E1: int = 50; N_E2: int = 90; N_T: int = 40; N_H: int = 30; GUARD: int = 8
    SCENE_SIM_MAX: float = 0.95
    GATE_AUC: float = 0.90                     # D1 halts below this on held-out real images
    SD: str = "stabilityai/stable-diffusion-3.5-medium"
    SEEDS: tuple = (0, 1)
    RANK: int = 16
    TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 2; GRAD_ACC: int = 2
    CAPTION: str = "a photograph, sks style"
    G_PER: int = 500; GEN_SEED_BASE: int = 880000
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    BLK: int = 8; LO: int = 1; HI: int = 5      # low/mid band, identical to E-LOWFREQ
    N_PERM: int = 10000; ALPHA: float = 0.01
    GRAD_CKPT: bool = False
C = CFG(); MEAS = C.CROP
CFG_SHA = hashlib.sha256(json.dumps({k:v for k,v in asdict(C).items() if k!="STAGES"},
                                    sort_keys=True, default=str).encode()).hexdigest()
ROOT = os.path.join(DRIVE, C.OUT)
for d in ("", "manifest","fingerprints","train_png","adapters","gens","csv","meta","figs"):
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)
CACHE = "/content/dx_cache"; os.makedirs(CACHE, exist_ok=True)
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False
def HALT(m): raise SystemExit(f"\n{'='*72}\n[HALT] {m}\n{'='*72}")

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],
               check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy.fft import dctn, idctn
from scipy import stats as sps
try:
    import peft.import_utils as _p1; _p1.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao as _p2; _p2.is_torchao_available = lambda: False
    print("[env] peft torchao dispatcher disabled")
except Exception: pass
Image.MAX_IMAGE_PIXELS = None
DEV = "cuda" if torch.cuda.is_available() else "cpu"
DT  = torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False
np.seterr(divide="raise", invalid="raise"); _MD = DEV
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN); print("[env] HF login ok")
else:
    print("[env] no HF token — D2/D3/D4 will fail on a gated model. Set HF_TOKEN.")
if DEV == "cuda":
    print(f"[env] {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB  dtype {DT}")

# ---- locate the Daxing root ----
def find_daxing():
    if C.DAXING:
        p = os.path.join(DRIVE, C.DAXING)
        if os.path.isdir(os.path.join(p, "image")): return p
        HALT(f"C.DAXING set but {p}/image not found")
    for depth in ("", "*/", "*/*/", "*/*/*/"):
        for cand in glob.glob(os.path.join(DRIVE, depth, "Daxing")):
            if os.path.isdir(os.path.join(cand, "image")): return cand
    HALT("could not find a Daxing folder containing image/ under your Drive. "
         "Set C.DAXING to its path relative to MyDrive.")
DX = find_daxing()
print(f"[env] Daxing root: {DX}")

def append_row(p,h,r):
    new = not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p, cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t = p+".tmp"; json.dump(o, open(t,"w"), **kw); os.replace(t,p)
def all_arms(): return [(f"{d}_s{s}", d, s) for s in C.SEEDS for d in C.DEVICES]
def all_tags(): return [t for t,_,_ in all_arms()]
def adir(t): return os.path.join(ROOT,"adapters",t)
def gdir(t): return os.path.join(ROOT,"gens",t)
def tdir(d): return os.path.join(ROOT,"train_png",d)
print(f"[cfg] SHA {CFG_SHA[:16]} | model {C.MODEL_NAME} | devices {C.DEVICES} | "
      f"angle {C.ANGLE} | {len(all_arms())} adapters")

In [ ]:
# %% 2 · forensic core — PRNU and the low/mid band, identical to the Dresden study
def _cp(fp): return os.path.join(CACHE, hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def _center(im):
    W,H = im.size
    if min(W,H) < MEAS: HALT(f"image smaller than the crop: {W}x{H} < {MEAS}")
    return im.crop(((W-MEAS)//2, (H-MEAS)//2, (W-MEAS)//2+MEAS, (H-MEAS)//2+MEAS))
def load_rgb(fp):
    # NOTE: no EXIF transpose. We want sensor-native pixels; auto-rotating would
    # destroy the spatial registration PRNU depends on.
    with Image.open(fp) as im:
        return np.asarray(_center(im.convert("RGB")), np.float32)
def load_lum(fp, cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    a = load_rgb(fp)
    l = (0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp), l)
    return l
def _c2(x,k):
    xt = torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt = torch.from_numpy(k)[None,None]
    if _MD == "cuda": return tF.conv2d(xt.cuda(), kt.cuda(), padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt, kt, padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0 = np.float32(4.0)
    co = pywt.wavedec2(img, "db8", level=4, mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch = ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v = np.maximum(_c2(ch*ch, np.ones((w,w),np.float32)/(w*w)) - s0, 0.0)
                vm = v if vm is None else np.minimum(vm, v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n = pywt.waverec2(out, "db8", mode="periodization")[:img.shape[0], :img.shape[1]]
    r = n - n.mean(); sd = float(r.std())
    if not np.isfinite(sd) or sd == 0: HALT("degenerate wavelet residual")
    return (r/sd).astype(np.float32)
_MASK = None
def band_mask():
    global _MASK
    if _MASK is None:
        m = np.zeros((C.BLK,C.BLK), np.float32)
        for i in range(C.BLK):
            for j in range(C.BLK):
                if C.LO <= i+j <= C.HI and not (i==0 and j==0): m[i,j] = 1.0
        _MASK = m; print(f"[band] {int(m.sum())} of {C.BLK*C.BLK} coefficients per block")
    return _MASK
def lowmid(Y):
    n = MEAS//C.BLK
    B = Y.reshape(n,C.BLK,n,C.BLK).transpose(0,2,1,3)
    D = dctn(B, axes=(2,3), norm="ortho") * band_mask()[None,None]
    R = idctn(D, axes=(2,3), norm="ortho").transpose(0,2,1,3).reshape(MEAS,MEAS).astype(np.float32)
    R = R - R.mean(); sd = float(R.std())
    if not np.isfinite(sd) or sd == 0: HALT("degenerate low/mid map")
    return (R/sd).astype(np.float32)
def zm(K):
    K = K - K.mean(axis=1, keepdims=True); return K - K.mean(axis=0, keepdims=True)
def wdft(K):
    F = np.fft.fft2(K.astype(np.float32)); mag = (np.abs(F)/K.shape[0]).astype(np.float32)
    sig = max(float(np.median(mag))/0.6745, 1e-12); s2 = np.float32(sig**2); vm=None
    for w in (3,5,7,9):
        k = np.ones((w,w),np.float32)/(w*w); mu = _c2(mag,k)
        v = np.maximum(_c2(mag*mag,k) - mu*mu, 0.0); vm = v if vm is None else np.minimum(vm,v)
    return np.real(np.fft.ifft2(F*(s2/(vm+s2)).astype(np.float32))).astype(np.float32)
def estK(files):
    num = den = None
    for fp in files:
        Y = load_lum(fp); W = wres(Y)
        num = W*Y if num is None else num + W*Y
        den = Y*Y if den is None else den + Y*Y
    return wdft(zm((num/np.maximum(den,1e-6)).astype(np.float32)))
def estL(files):
    acc = None
    for fp in files:
        m = lowmid(load_lum(fp)); acc = m if acc is None else acc + m
    return wdft(zm((acc/len(files)).astype(np.float32)))
def ncc(a,b):
    a = a-a.mean(); b = b-b.mean()
    na = float(np.linalg.norm(a)); nb = float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def nccb(a,B):
    a = a-a.mean(); na = a.norm()
    Bc = B - B.mean(dim=(1,2), keepdim=True); nb = Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
def thumb(fp, n=64):
    with Image.open(fp) as im:
        return np.asarray(im.convert("L").resize((n,n), Image.BILINEAR), np.float32).ravel()
def auc_of(same, cross):
    lab = np.r_[np.ones(len(same)), np.zeros(len(cross))]; sc = np.r_[same, cross]
    o = np.argsort(sc); rk = np.empty(len(sc)); rk[o] = np.arange(len(sc))
    return float((rk[lab==1].mean() - (lab.sum()-1)/2) / (len(sc) - lab.sum()))
print("[core] ready")

In [ ]:
# %% 3 · D0 — per-device scene clustering, cross-device alignment, STRATIFIED splits
#
# Two failed attempts led here, and both failures were informative.
#
#   Attempt 1, per-device clustering + whole-scene splits: each device got its own scene->split
#   map, so one device's H held textured scenes while another's held sky. That produced an 8x
#   spread in the per-device real contrast (21x for low/mid) against 1.27x on Dresden.
#
#   Attempt 2, joint clustering: k-means on raw thumbnails grouped by DEVICE as much as by
#   scene -- several clusters contained images from only one or two devices. Whole-scene
#   capacity collapsed to 70 images against the 210 needed.
#
# What this does instead. Cluster each device separately (which respects its own colour and
# exposure), then align the cluster labels ACROSS devices by matching centroids. Every split
# then takes a proportional slice of every scene, with each device contributing the same count
# per scene. Composition is identical across splits and across devices, so D1's per-device
# contrast becomes a comparison of devices rather than of scene assignments.
#
# Scene sharing between E2 and H is deliberate and safe here: structure common to all five
# devices correlates with every fingerprint equally and cancels in the paired contrast, which
# is also why RESIDUALISE exists. The residual risk is a repeated FRAME, and that is what the
# EXIF burst check below actually tests.
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","scikit-learn","ImageHash"], check=False)
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
from datetime import datetime

def exif_time(p):
    """EXIF DateTimeOriginal as a timestamp, or None. Two frames of one burst share it;
    two different photographs of a blank wall do not. Appearance-based metrics cannot tell
    those apart on this dataset -- both raw thumbnail cosine and perceptual hashing saturate on flat
    scenes, which is why neither is used here."""
    try:
        with Image.open(p) as im:
            ex = im._getexif() or {}
        v = ex.get(36867) or ex.get(306)
        return datetime.strptime(v, "%Y:%m:%d %H:%M:%S").timestamp() if v else None
    except Exception:
        return None

N_SCENES   = 8         # lower to 6 for larger, better-balanced scenes and bigger splits
BURST_SECS = 3         # EXIF capture times closer than this across splits = burst frames
MIN_SPLIT  = {"E1":30, "E2":60, "T":30, "H":20}

def dev_dir(dev, angle):
    for pat in (os.path.join(DX,"image","*",dev,angle), os.path.join(DX,"image",dev,angle),
                os.path.join(DX,"image","*",dev),       os.path.join(DX,"image",dev)):
        hits = [p for p in glob.glob(pat) if os.path.isdir(p)]
        if hits: return sorted(hits)[0]
    return None

def jpgs(d):
    if d is None: return []
    return sorted(p for p in glob.glob(os.path.join(d,"*"))
                  if p.lower().endswith((".jpg",".jpeg"))
                  and os.path.basename(p) != "desktop.ini" and ".tmp.driveupload" not in p)

def _norm_rows(X):
    X = X - X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True); s[s < 1e-6] = 1e-6
    return X / s

def d0():
    print("[D0] split builder v7 — held-out scenes for H, exact-fit allocation\n")
    print(f"[D0] scanning {DX}\n")
    inv = {}
    for dev in list(C.DEVICES) + list(C.CROSS_MODEL):
        row = {a or "flat": jpgs(dev_dir(dev, a)) for a in ("0","90","180","")}
        inv[dev] = {k:v for k,v in row.items() if v}
        print(f"  {dev}: " + (", ".join(f"{k}={len(v)}" for k,v in inv[dev].items()) or "NONE"))

    targets = {"E1":C.N_E1, "E2":C.N_E2, "T":C.N_T, "H":C.N_H}
    files = {}
    for dev in C.DEVICES:
        f = inv[dev].get(C.ANGLE, [])
        if len(f) < sum(targets.values()):
            HALT(f"{dev} has {len(f)} in angle '{C.ANGLE}', needs {sum(targets.values())}")
        files[dev] = f
    sizes = {}
    for dev in C.DEVICES:
        cnt = Counter()
        for p in files[dev][:40]:
            try:
                with Image.open(p) as im: cnt[im.size] += 1
            except Exception: pass
        sizes[dev] = cnt.most_common(1)[0][0]
    print(f"\n[D0] resolutions: {sizes}")
    if len(set(sizes.values())) != 1: HALT(f"devices disagree on resolution: {sizes}")
    if min(min(s) for s in sizes.values()) < MEAS: HALT("resolution below the crop")

    # ---- 1. cluster each device on its own, then align labels across devices ----
    print(f"\n[D0] clustering each device into {N_SCENES} scenes, then aligning labels ...")
    lab, cen = {}, {}
    for dev in C.DEVICES:
        T = _norm_rows(np.stack([thumb(p) for p in files[dev]]))
        km = KMeans(n_clusters=N_SCENES, n_init=10, random_state=0).fit(T)
        lab[dev] = km.labels_; cen[dev] = km.cluster_centers_
    ref = C.DEVICES[0]; remap = {ref: {i:i for i in range(N_SCENES)}}
    for dev in C.DEVICES[1:]:
        A = cen[ref]/np.linalg.norm(cen[ref],axis=1,keepdims=True)
        B = cen[dev]/np.linalg.norm(cen[dev],axis=1,keepdims=True)
        r, c = linear_sum_assignment(-(A @ B.T))          # maximise centroid similarity
        remap[dev] = {int(cj): int(ri) for ri, cj in zip(r, c)}
    scene = {dev: {} for dev in C.DEVICES}
    for dev in C.DEVICES:
        for p, l in zip(files[dev], lab[dev]):
            scene[dev].setdefault(remap[dev][int(l)], []).append(p)

    counts = {s: [len(scene[d].get(s, [])) for d in C.DEVICES] for s in range(N_SCENES)}
    print(f"[D0] aligned scene sizes (rows = scene, columns = {list(C.DEVICES)}):")
    for s in range(N_SCENES):
        row = counts[s]; bal = min(row)/max(max(row),1)
        print(f"     scene {s}: {row}   min {min(row):3d}   balance {bal:.2f}")

    # ---- 2. HELD-OUT SCENES for H, shared scenes for E1/E2/T ----
    #
    # The burst guard showed that within a scene nearly every image is under BURST_SECS from
    # its neighbour: the scene IS one continuous capture session of ~30 shots. So burst-level
    # separation does not exist in this data, and a time guard simply consumed the pool.
    #
    # The separation that actually matters is E2 (fingerprint estimation) vs H (query). If an
    # H image shares a capture session with something in its OWN device's E2, the own-device
    # correlation is inflated while the other devices' is not, because they shot that scene at
    # a different moment. R_real then inflates and lambda_U = U/R_real looks tighter than it is.
    #
    # E1 and T may share scenes with E2 without harm: E1 only feeds the split-half reliability
    # check, and T trains adapters whose generations come from text prompts, with memorisation
    # covered separately by the copy audit.
    cap = {s_: min(counts[s_]) for s_ in range(N_SCENES)}
    order = sorted(cap, key=lambda x: cap[x])                 # smallest scenes first
    Q, qcap = [], 0
    for s_ in order:                                          # hold back scenes for H
        if qcap >= targets["H"]: break
        Q.append(s_); qcap += cap[s_]
    F = [s_ for s_ in range(N_SCENES) if s_ not in Q]
    fcap = sum(cap[s_] for s_ in F)
    print(f"\n[D0] scene capacity {cap}")
    print(f"     held-out scenes for H : {sorted(Q)}  capacity {qcap}  (target {targets['H']})")
    print(f"     shared scenes E1/E2/T : {sorted(F)}  capacity {fcap}  "
          f"(target {targets['E1']+targets['E2']+targets['T']})")
    if qcap < MIN_SPLIT["H"]:
        HALT(f"held-out pool {qcap} below the floor {MIN_SPLIT['H']} for H")
    need_F = targets["E1"] + targets["E2"] + targets["T"]
    if fcap < need_F:
        floors = sum(MIN_SPLIT[k] for k in ("E1","E2","T"))
        if fcap < floors:
            HALT(f"shared pool {fcap} is below the floors {floors} "
                 f"{ {k:MIN_SPLIT[k] for k in ('E1','E2','T')} }.\n"
                 f"Try N_SCENES=6 for coarser, better-balanced scenes, or drop device 1105 "
                 f"(it pins every scene minimum).")
        sc = fcap/need_F
        for k in ("E1","E2","T"): targets[k] = max(MIN_SPLIT[k], int(targets[k]*sc))
        # rounding plus the floors can overshoot by a few images; trim from whichever split
        # sits furthest above its own floor until the allocation fits exactly.
        while targets["E1"]+targets["E2"]+targets["T"] > fcap:
            k = max(("E1","E2","T"), key=lambda x: targets[x]-MIN_SPLIT[x])
            if targets[k] <= MIN_SPLIT[k]:
                HALT(f"cannot fit {fcap} to the floors — this should be unreachable")
            targets[k] -= 1
        print(f"     scaled E1/E2/T to fit: "
              f"{ {k:targets[k] for k in ('E1','E2','T')} }  "
              f"(sum {targets['E1']+targets['E2']+targets['T']} of {fcap})")
    targets["H"] = min(targets["H"], qcap)
    if targets["E2"] < 80:
        print(f"     NOTE E2={targets['E2']}: the earlier run reached residualised split-half "
              f"0.4034 at E2=90.\n          Fingerprint reliability will be lower here. "
              f"N_SCENES=6 usually recovers some of it.")
    tf = targets["E1"] + targets["E2"] + targets["T"]
    alloc = {}
    for s_ in F:                                              # stratify E1/E2/T over F
        base = {k: int(cap[s_]*targets[k]/tf) for k in ("E1","E2","T")}
        rem  = cap[s_] - sum(base.values())
        fr   = sorted(("E1","E2","T"), key=lambda k: -((cap[s_]*targets[k]/tf) - base[k]))
        for n_ in range(rem): base[fr[n_ % 3]] += 1
        alloc[s_] = {**base, "H": 0}
    for s_ in Q:                                              # H takes the held-out scenes
        alloc[s_] = {"E1":0, "E2":0, "T":0, "H": cap[s_]}

    # EXIF timestamp cache. Sorting each scene's pool by capture time keeps the images a
    # split receives contiguous in time, which is tidier than filename order and costs
    # nothing. Correctness no longer depends on it: E2 and H are scene-disjoint, so no
    # capture session can span them.
    TCACHE = {}
    def _ts(p):
        if p not in TCACHE: TCACHE[p] = exif_time(p)
        return TCACHE[p]

    hdr = ["dev","split","idx","path"]; cp = os.path.join(ROOT,"manifest","manifest.csv")
    if os.path.exists(cp): os.remove(cp)
    man = {"config_sha":CFG_SHA,"root":DX,"model":C.MODEL_NAME,"angle":C.ANGLE,"meas":MEAS,
           "resolution":list(next(iter(sizes.values()))),
           "split_method":"per-device k-means, cross-device centroid alignment, stratified",
           "n_scenes":N_SCENES,"targets":targets,"scene_capacity":cap,
           "scene_counts":{str(k):v for k,v in counts.items()},
           "allocation":{str(s):alloc[s] for s in alloc},"splits":{}}
    min_gap_s, flagged = 1e9, []
    for dev in C.DEVICES:
        sp = {k: [] for k in targets}
        for s_ in range(N_SCENES):
            pool = sorted(scene[dev].get(s_, []), key=lambda p: (_ts(p) or 0, p))[:cap[s_]]
            i2 = 0
            for k in ("E1","E2","T","H"):
                n2 = alloc[s_][k]; sp[k].extend(pool[i2:i2+n2]); i2 += n2
        short = {k: targets[k]-len(v) for k,v in sp.items() if len(v) < targets[k]}
        if short: HALT(f"{dev}: {short} short of {targets}")
        sp = {k: v[:targets[k]] for k,v in sp.items()}
        # HARD GATE: no path may appear in two splits.
        seen = {}
        for k, v in sp.items():
            for p in v:
                if p in seen:
                    HALT(f"{dev}: {os.path.basename(p)} appears in both {seen[p]} and {k}")
                seen[p] = k
        # INFORMATIONAL: EXIF capture-time proximity across splits, i.e. burst frames.
        TS = {k: [(p, exif_time(p)) for p in v[::max(1,len(v)//12)][:12]] for k,v in sp.items()}
        have = sum(1 for k in TS for _,t in TS[k] if t is not None)
        for a,b in itertools.combinations(sp, 2):
            ta = [t for _,t in TS[a] if t is not None]; tb = [t for _,t in TS[b] if t is not None]
            if not ta or not tb: continue
            g = min(abs(x-y) for x in ta for y in tb)
            min_gap_s = min(min_gap_s, g)
            if g < BURST_SECS: flagged.append((dev, a, b, round(g,1)))
        for k,v in sp.items():
            for j,fp in enumerate(v): append_row(cp, hdr, [dev,k,j,fp])
        man["splits"][dev] = {k:len(v) for k,v in sp.items()}
        print(f"  {dev}: {man['splits'][dev]}")
    for dev in C.CROSS_MODEL:
        for j,fp in enumerate(inv[dev].get(C.ANGLE, [])[:targets['E2']]):
            append_row(cp, hdr, [f"X_{dev}","E2",j,fp])
    man["audit"] = {"smallest_cross_split_capture_gap_s":
                    (None if min_gap_s>=1e9 else min_gap_s),"burst_pairs":flagged,
                    "burst_threshold_s":BURST_SECS}
    ajson(man, os.path.join(ROOT,"manifest","manifest.json"), indent=2)

    print(f"\n[D0] no path appears in two splits (hard gate passed).")
    if min_gap_s >= 1e9:
        print("     EXIF capture times unavailable — burst check skipped.")
    else:
        print(f"     smallest cross-split EXIF capture gap: {min_gap_s:.0f} s "
              f"(burst threshold {BURST_SECS} s)")
        if flagged:
            print(f"     {len(flagged)} split pair(s) contain captures within {BURST_SECS}s "
                  f"of each other:")
            for r in flagged[:8]: print(f"       {r}")
            print("     Any pair involving H would be a real problem — H draws from scenes\n"
                  "     that E1/E2/T never touch, so a shared capture time there means the\n"
                  "     scene alignment is wrong. Pairs among E1/E2/T alone are expected and\n"
                  "     harmless: they share scenes by design and none of them is the query set.")
    print(f"\n[D0] PASS — identical scene composition in every split, for every device.")
    print(f"     Scene structure shared between E2 and H is common to all five devices and "
          f"cancels\n     in the paired contrast; RESIDUALISE removes it explicitly. Watch the "
          f"per-device\n     spread in D1: it should now be much closer to Dresden's 1.27x than "
          f"to 8x.")
if stage("D0"): d0()


In [ ]:
# %% 4 · D1 — fingerprints, residualisation, gate, orientation test
#
# COMPLETE CELL. Replaces every earlier D1. Four things are folded in, and losing any one of
# them has already cost a run:
#
#   1. MANIFEST STAMPING. Fingerprints are tied to a hash of the manifest. D1 used to skip
#      re-estimation whenever the .npy existed, so changing the splits silently reused
#      fingerprints built from the old ones — which broke the E2/H scene disjointness that
#      D0 exists to provide.
#   2. STRING DEVICE CODES. "1101" is numeric, so pandas reads the column as int64 and every
#      `df.F_dev == "1101"` matched nothing, giving empty slices and a divide-by-zero.
#   3. RESIDUALISATION. Raw same-model pairwise NCC is ~0.46 against Dresden's 0.0073, because
#      every device shot the same eight scenes. Leave-one-out projection removes it.
#   4. THE RIGHT GATE. An unpaired same-vs-cross AUC is inflated on both sides by that shared
#      component. The gate is 5-way top-1 identification and the sign of the paired contrast.
RESIDUALISE = True
GATE_TOP1   = 0.80

DEVCOLS = {"dev","img_dev","F_dev","K","L","tag","role","split"}
def rd(p, **kw):
    df = pd.read_csv(p, **kw)
    for c in df.columns:
        if c in DEVCOLS: df[c] = df[c].astype(str)
    return df

def MANIF():
    M = {}
    for r in _csv.DictReader(open(os.path.join(ROOT,"manifest","manifest.csv"))):
        M.setdefault((str(r["dev"]), r["split"]), []).append(r["path"])
    return M

def _manifest_stamp():
    """Hash of the splits AND of RESIDUALISE. The positive controls are computed against
    whichever fingerprints were in force, so flipping RESIDUALISE invalidates them just as
    surely as changing the splits does. Stamping only the manifest let a raw-fingerprint
    positive control survive into a residualised run -- visible as top-1 1.000 alongside an
    unpaired AUC of 0.67, which is the signature of a shared per-image offset."""
    h = hashlib.sha256(open(os.path.join(ROOT,"manifest","manifest.csv"),"rb").read())
    h.update(f"|RESIDUALISE={RESIDUALISE}".encode())
    return h.hexdigest()[:16]

def _resid(K, others):
    """Remove the component of K along the mean of the other devices' fingerprints."""
    Mn = np.mean(others, axis=0)
    a  = Mn - Mn.mean(); a /= (np.linalg.norm(a) + 1e-30)
    Kc = K - K.mean()
    return (Kc - float((Kc*a).sum())*a).astype(np.float32)

def d1():
    print("[D1] fingerprints v4 — stamp covers RESIDUALISE, string-safe, top-1 gate\n")
    M = MANIF()

    stamp = _manifest_stamp()
    spf   = os.path.join(ROOT,"fingerprints","_manifest_stamp.txt")
    prev  = open(spf).read().strip() if os.path.exists(spf) else None
    if prev != stamp:
        n = m = 0
        for p in glob.glob(os.path.join(ROOT,"fingerprints","*.npy")): os.remove(p); n += 1
        for p in glob.glob(os.path.join(ROOT,"csv","d1_positive_*.csv")): os.remove(p); m += 1
        for p in glob.glob(os.path.join(ROOT,"fingerprints","*.json")):
            if "orientation" not in p: os.remove(p)
        open(spf,"w").write(stamp)
        print(f"[D1] stamp changed ({prev} -> {stamp}); removed {n} fingerprint(s) and "
              f"{m} positive control(s).\n")
    else:
        print(f"[D1] stamp unchanged ({stamp}); reusing cached fingerprints and controls.\n")

    for dev in C.DEVICES:
        for which in ("E1","E2"):
            for tag, fn in (("K",estK), ("L",estL)):
                p = os.path.join(ROOT,"fingerprints",f"{tag}_{dev}_{which}.npy")
                if os.path.exists(p): continue
                t0 = time.time(); np.save(p, fn(M[(dev,which)]))
                print(f"[D1] {tag}_{dev}_{which} from {len(M[(dev,which)])} imgs "
                      f"({time.time()-t0:.0f}s)")
    for dev in C.CROSS_MODEL:
        p = os.path.join(ROOT,"fingerprints",f"K_X{dev}_E2.npy")
        if not os.path.exists(p) and (f"X_{dev}","E2") in M:
            np.save(p, estK(M[(f"X_{dev}","E2")])); print(f"[D1] cross-model K_X{dev}")

    raw  = {t: {d: np.load(os.path.join(ROOT,"fingerprints",f"{t}_{d}_E2.npy"))
                for d in C.DEVICES} for t in ("K","L")}
    raw1 = {t: {d: np.load(os.path.join(ROOT,"fingerprints",f"{t}_{d}_E1.npy"))
                for d in C.DEVICES} for t in ("K","L")}
    diag = {}
    for t in ("K","L"):
        sh = [ncc(raw1[t][d], raw[t][d]) for d in C.DEVICES]
        pw = [ncc(raw[t][a], raw[t][b]) for a,b in itertools.combinations(C.DEVICES,2)]
        diag[t] = {"split_half_raw":float(np.mean(sh)), "pairwise_raw":float(np.mean(pw))}
        print(f"\n[D1] {t} RAW   split-half {np.mean(sh):+.4f} | pairwise {np.mean(pw):+.4f}"
              f" | device-specific fraction ~{np.mean(sh)-np.mean(pw):+.4f}")

    F, F1 = {t: dict(raw[t]) for t in ("K","L")}, {t: dict(raw1[t]) for t in ("K","L")}
    if RESIDUALISE:
        for t in ("K","L"):
            F[t]  = {d: _resid(raw[t][d],  [raw[t][o]  for o in C.DEVICES if o!=d])
                     for d in C.DEVICES}
            F1[t] = {d: _resid(raw1[t][d], [raw1[t][o] for o in C.DEVICES if o!=d])
                     for d in C.DEVICES}
            sh = [ncc(F1[t][d], F[t][d]) for d in C.DEVICES]
            pw = [ncc(F[t][a], F[t][b]) for a,b in itertools.combinations(C.DEVICES,2)]
            diag[t].update({"split_half_resid":float(np.mean(sh)),
                            "pairwise_resid":float(np.mean(pw))})
            print(f"[D1] {t} RESID split-half {np.mean(sh):+.4f} | pairwise {np.mean(pw):+.4f}"
                  f"   (the -1/(k-1) = {-1/(len(C.DEVICES)-1):+.2f} floor means a clean removal)")
            for d in C.DEVICES:
                np.save(os.path.join(ROOT,"fingerprints",f"{t}R_{d}_E2.npy"), F[t][d])

    op = os.path.join(ROOT,"fingerprints","orientation_test.json")
    if not os.path.exists(op):
        dv = C.DEVICES[0]
        f0, f180 = jpgs(dev_dir(dv,"0"))[:60], jpgs(dev_dir(dv,"180"))[:60]
        if len(f0) >= 30 and len(f180) >= 30:
            r = {"device":dv, "ncc_within_0":ncc(estK(f0[:15]), estK(f0[15:30])),
                 "ncc_0_vs_180":ncc(estK(f0[:30]), estK(f180[:30]))}
            ajson(r, op, indent=2)
            print(f"\n[D1] orientation test on {dv}: within-'0' {r['ncc_within_0']:+.4f}, "
                  f"'0' vs '180' {r['ncc_0_vs_180']:+.4f}")

    for tag, mapf in (("K",wres), ("L",lowmid)):
        p = os.path.join(ROOT,"csv",f"d1_positive_{tag}.csv")
        if os.path.exists(p): continue
        for d in C.DEVICES:
            for i, fp in enumerate(M[(d,"H")]):
                Y = load_lum(fp); W = mapf(Y)
                for k_ in C.DEVICES:
                    append_row(p, ["img_dev","F_dev","idx","rho"],
                               [d, k_, i, f"{ncc(W, Y*F[tag][k_]):.6e}"])
            print(f"[D1] positive control {tag} {d}")

    gates = {}
    for tag in ("K","L"):
        pc = rd(os.path.join(ROOT,"csv",f"d1_positive_{tag}.csv"))
        missing = [d for d in C.DEVICES if not len(pc[pc.img_dev==d])]
        if missing: HALT(f"positive control has no rows for {missing} — device codes mismatch")
        auc = auc_of(pc[pc.img_dev==pc.F_dev]["rho"].to_numpy(),
                     pc[pc.img_dev!=pc.F_dev]["rho"].to_numpy())
        piv  = pc.pivot_table(index=["img_dev","idx"], columns="F_dev", values="rho")
        top1 = float((piv.idxmax(axis=1).to_numpy() ==
                      piv.index.get_level_values("img_dev").to_numpy()).mean())
        per = []
        for d in C.DEVICES:
            g   = pc[pc.img_dev==d]
            own = g[g.F_dev==d].sort_values("idx")["rho"].to_numpy()
            oth = np.mean([g[g.F_dev==o].sort_values("idx")["rho"].to_numpy()
                           for o in C.DEVICES if o!=d], axis=0)
            n = min(len(own), np.size(oth))
            if n == 0: HALT(f"empty slice for {d} in the {tag} positive control")
            per.append(float((own[:n]-oth[:n]).mean()))
        per = np.array(per); R = float(per.mean())
        se  = float(per.std(ddof=1)/np.sqrt(len(per)))
        gates[tag] = {**diag[tag], "auc_unpaired":auc, "top1":top1, "R_real":R, "se":se,
                      "R_real_lower99":float(R - sps.norm.ppf(0.99)*se),
                      "t_paired":float(R/se if se>0 else np.inf),
                      "per_device":per.tolist(), "spread":float(per.max()/max(per.min(),1e-12)),
                      "residualised":RESIDUALISE, "n_E2":len(M[(C.DEVICES[0],"E2")])}
        print(f"\n[D1] {tag}: 5-way top-1 {top1:.3f} (chance {1/len(C.DEVICES):.2f}) | "
              f"unpaired AUC {auc:.4f} (context only)")
        if RESIDUALISE and top1 > 0.95 and auc < 0.85:
            print(f"     *** top-1 {top1:.3f} with AUC {auc:.3f} means a large shared per-image "
                  f"offset:\n         within an image the right device wins, but the pooled "
                  f"distributions overlap.\n         With RESIDUALISE on that should not happen. "
                  f"The controls were probably built\n         against raw fingerprints — delete "
                  f"csv/d1_positive_*.csv and re-run D1.")
        print(f"     R_real {R:.5e}  lower99 {gates[tag]['R_real_lower99']:.5e}  "
              f"t {gates[tag]['t_paired']:+.2f}")
        print(f"     per-device {np.array2string(per, precision=5)}   "
              f"spread {gates[tag]['spread']:.2f}x   (Dresden 1.27x, Kodak 2.7x)")
    ajson(gates, os.path.join(ROOT,"fingerprints","d1_gates.json"), indent=2)

    K = gates["K"]
    print("\n" + "="*72)
    if K["top1"] < GATE_TOP1 or K["R_real"] <= 0:
        HALT(f"PRNU GATE FAILED: 5-way top-1 {K['top1']:.3f}, paired contrast "
             f"{K['R_real']:.3e}.\nDevice identity is not recoverable from this device class at "
             f"a {MEAS}px crop. That is a\nlegitimate finding — report it. Do NOT tune the crop "
             f"or extractor to rescue it.")
    print(f"GATE PASSED: 5-way top-1 {K['top1']:.3f}, paired contrast {K['R_real']:.3e} "
          f"(t {K['t_paired']:+.1f}), E2 = {K['n_E2']} images.")
    print(f"Report BOTH pairwise numbers: raw {diag['K']['pairwise_raw']:+.3f} against "
          f"Dresden's 0.0073,\nresidualised {diag['K'].get('pairwise_resid', float('nan')):+.3f}. "
          f"Every device shot the same scenes, so the\nfingerprints share camera-model structure; "
          f"the paired statistic cancels it either way.")
    if gates["L"]["top1"] < 0.90:
        print(f"\nNOTE low/mid top-1 is {gates['L']['top1']:.3f} at E2={K['n_E2']}. Below ~0.90 "
              f"that arm is too\n     noisy to carry a bound on this dataset.")
    print("="*72)
if stage("D1"): d1()


In [ ]:
# %% 5 · D2 — autoencoder round-trip retention, both representations
#
# COMPLETE CELL. Three defects in the previous version, each already met elsewhere here:
#   1. It loaded RAW fingerprints while D1's gate used RESIDUALISED ones. The paired contrast
#      does not fully cancel the shared camera-model component — R_real moved 3.875e-02 ->
#      5.727e-02 when D1's controls were rebuilt residualised — so eta would not have been
#      comparable to anything D1 reported.
#   2. Device codes are numeric strings; pandas reads them as int64 and `d.dev == "1101"`
#      matches nothing, giving empty slices and a divide-by-zero.
#   3. The cached CSV was keyed on existence alone, so it survived a fingerprint change.
#
# Requires D1's cell to have been executed (it defines rd, MANIF and _manifest_stamp).
for _n in ("rd","MANIF","_manifest_stamp"):
    if _n not in globals(): HALT(f"run the D1 cell first — it defines {_n}()")

def d2():
    print("[D2] round-trip retention v2 — residualised fingerprints, string-safe, stamped\n")
    from diffusers import AutoencoderKL
    M = MANIF()

    # use exactly the fingerprints D1's gate used
    resid = all(os.path.exists(os.path.join(ROOT,"fingerprints",f"{t}R_{d}_E2.npy"))
                for t in ("K","L") for d in C.DEVICES)
    pre_ = (lambda t: f"{t}R") if resid else (lambda t: t)
    print(f"[D2] fingerprints: {'RESIDUALISED' if resid else 'RAW'} "
          f"({pre_('K')}_*_E2.npy)")
    if not resid:
        print("     WARNING: residualised fingerprints absent. Re-run D1 with RESIDUALISE=True,\n"
              "     or eta will be measured against a different quantity than D1's R_real.")
    FP = {t: {d: np.load(os.path.join(ROOT,"fingerprints",f"{pre_(t)}_{d}_E2.npy"))
              for d in C.DEVICES} for t in ("K","L")}

    stamp = _manifest_stamp() + ("|resid" if resid else "|raw")
    sf = os.path.join(ROOT,"csv","_d2_stamp.txt")
    p  = os.path.join(ROOT,"csv","d2_vae.csv")
    if os.path.exists(p) and (not os.path.exists(sf) or open(sf).read().strip() != stamp):
        os.remove(p); print("[D2] stamp changed — discarded the cached round-trip CSV")

    if not os.path.exists(p):
        vae = AutoencoderKL.from_pretrained(C.SD, subfolder="vae",
                                            torch_dtype=torch.float32, token=True).to(DEV).eval()
        try:
            for dv in C.DEVICES:
                t0 = time.time()
                for i, fp in enumerate(M[(dv,"H")]):
                    rgb = load_rgb(fp)
                    with torch.no_grad():
                        x   = torch.from_numpy(rgb/127.5-1.0).permute(2,0,1)[None].to(DEV)
                        rec = vae.decode(vae.encode(x).latent_dist.mode()).sample[0]
                        rec = rec.permute(1,2,0).cpu().numpy()
                    Rimg = np.clip((rec+1.0)*127.5, 0, 255).astype(np.float32)
                    for st, img in (("pre",rgb), ("post",Rimg)):
                        Y  = (0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]).astype(np.float32)
                        WK, WL = wres(Y), lowmid(Y)
                        for k_ in C.DEVICES:
                            append_row(p, ["dev","stage","idx","F_dev","rho_K","rho_L"],
                                       [dv, st, i, k_,
                                        f"{ncc(WK, Y*FP['K'][k_]):.6e}",
                                        f"{ncc(WL, Y*FP['L'][k_]):.6e}"])
                print(f"[D2] {dv}: {len(M[(dv,'H')])} images ({time.time()-t0:.0f}s)")
        finally:
            del vae; gc.collect(); torch.cuda.empty_cache()
        open(sf,"w").write(stamp)

    df = rd(p)                                     # forces the device columns to str
    for c_ in ("rho_K","rho_L"): df[c_] = df[c_].astype(float)
    rng = np.random.default_rng(0)
    out = {"residualised": resid, "n_H": int(len(M[(C.DEVICES[0],"H")]))}

    for tag, cname in (("K","rho_K"), ("L","rho_L")):
        def contrasts(st):
            per = {}
            for dv in C.DEVICES:
                g   = df[(df.dev==dv) & (df.stage==st)]
                own = g[g.F_dev==dv].sort_values("idx")[cname].to_numpy(float)
                oth = np.mean([g[g.F_dev==o].sort_values("idx")[cname].to_numpy(float)
                               for o in C.DEVICES if o!=dv], axis=0)
                n = min(len(own), np.size(oth))
                if n == 0: HALT(f"empty slice for {dv} at stage {st} — device codes mismatch")
                per[dv] = (own[:n]-oth[:n])
            return per
        pre, post = contrasts("pre"), contrasts("post")
        Rpre  = float(np.mean([v.mean() for v in pre.values()]))
        Rpost = float(np.mean([v.mean() for v in post.values()]))
        eta   = Rpost/Rpre

        # paired image-level bootstrap: resample the SAME held-out images at both stages
        boots = []
        for _ in range(10000):
            a = np.mean([pre[dv][rng.integers(0,len(pre[dv]),len(pre[dv]))].mean()
                         for dv in C.DEVICES])
            idx = {dv: rng.integers(0,len(post[dv]),len(post[dv])) for dv in C.DEVICES}
            b = np.mean([post[dv][idx[dv]].mean() for dv in C.DEVICES])
            boots.append(b/a)
        lo, hi = np.percentile(boots, [2.5, 97.5])

        aucs = {st: auc_of(df[(df.stage==st)&(df.dev==df.F_dev)][cname].to_numpy(float),
                           df[(df.stage==st)&(df.dev!=df.F_dev)][cname].to_numpy(float))
                for st in ("pre","post")}
        perdev = {dv: float(post[dv].mean()/pre[dv].mean()) for dv in C.DEVICES}
        sprd   = max(perdev.values())/max(min(perdev.values()), 1e-12)
        out[tag] = {"R_pre":Rpre, "R_post":Rpost, "eta":eta, "eta_ci":[float(lo),float(hi)],
                    "auc_pre":aucs["pre"], "auc_post":aucs["post"],
                    "eta_per_device":perdev, "eta_spread":float(sprd)}
        print(f"\n[D2] {tag}: R pre {Rpre:.5e} -> post {Rpost:.5e}")
        print(f"     eta {eta:.4f}  95% CI [{lo:.4f}, {hi:.4f}]   "
              f"AUC {aucs['pre']:.4f} -> {aucs['post']:.4f}")
        print(f"     per-device eta { {k: round(v,3) for k,v in perdev.items()} }  "
              f"spread {sprd:.2f}x")
    ajson(out, os.path.join(ROOT,"d2_vae.json"), indent=2)

    print("\n" + "="*72)
    print(f"Dresden reference: PRNU eta 0.3661 (AUC 1.000 -> 0.9814), 95% CI [0.344, 0.387]")
    print(f"                   low/mid eta 0.5544 (AUC 0.8433)")
    print(f"This run:          PRNU eta {out['K']['eta']:.4f}, low/mid {out['L']['eta']:.4f}")
    print("A HIGHER eta here would mean a modern smartphone ISP passes MORE device signal")
    print("through the latent autoencoder than 2010 CCD, not less. Either direction is a")
    print("result; neither is a failure. Report the CI, not the point estimate alone.")
    print("="*72)
if stage("D2"): d2()


In [ ]:
# %% 5b · D2b — eta robustness: does residualisation change it?
#
# eta is now load-bearing: 0.5601 [0.4742, 0.6588] against Dresden's 0.3661 [0.344, 0.387],
# intervals not overlapping. But Daxing's eta was measured with RESIDUALISED fingerprints and
# Dresden's with plain ones (Dresden needed no residualisation -- its kappa was already 0.0073).
# eta is a ratio of paired contrasts using the same fingerprints at both stages, so the shared
# component SHOULD cancel. That is an assumption, not a measurement. This cell measures it.
#
# NON-DESTRUCTIVE. Writes csv/d2_vae_raw.csv and d2_vae_raw.json; touches no stamp and
# overwrites nothing from D1 or D2. Roughly 3 minutes -- one VAE pass over the 150 held-out
# images, correlated against the RAW fingerprints D1 already saved alongside the residualised
# ones. Run it before D3.
for _n in ("rd","MANIF"):
    if _n not in globals(): HALT(f"run the D1 cell first — it defines {_n}()")
if not os.path.exists(os.path.join(ROOT,"d2_vae.json")):
    HALT("run D2 first — this compares against its result")

def d2b():
    from diffusers import AutoencoderKL
    M = MANIF()
    missing = [f"{t}_{d}_E2.npy" for t in ("K","L") for d in C.DEVICES
               if not os.path.exists(os.path.join(ROOT,"fingerprints",f"{t}_{d}_E2.npy"))]
    if missing: HALT(f"raw fingerprints absent: {missing[:3]} — re-run D1")
    FP = {t: {d: np.load(os.path.join(ROOT,"fingerprints",f"{t}_{d}_E2.npy"))
              for d in C.DEVICES} for t in ("K","L")}
    print("[D2b] eta with RAW fingerprints (D2 used residualised)\n")

    p = os.path.join(ROOT,"csv","d2_vae_raw.csv")
    if not os.path.exists(p):
        vae = AutoencoderKL.from_pretrained(C.SD, subfolder="vae",
                                            torch_dtype=torch.float32, token=True).to(DEV).eval()
        try:
            for dv in C.DEVICES:
                t0 = time.time()
                for i, fp in enumerate(M[(dv,"H")]):
                    rgb = load_rgb(fp)
                    with torch.no_grad():
                        x   = torch.from_numpy(rgb/127.5-1.0).permute(2,0,1)[None].to(DEV)
                        rec = vae.decode(vae.encode(x).latent_dist.mode()).sample[0]
                        rec = rec.permute(1,2,0).cpu().numpy()
                    Rimg = np.clip((rec+1.0)*127.5, 0, 255).astype(np.float32)
                    for st, img in (("pre",rgb), ("post",Rimg)):
                        Y = (0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]).astype(np.float32)
                        WK, WL = wres(Y), lowmid(Y)
                        for k_ in C.DEVICES:
                            append_row(p, ["dev","stage","idx","F_dev","rho_K","rho_L"],
                                       [dv, st, i, k_,
                                        f"{ncc(WK, Y*FP['K'][k_]):.6e}",
                                        f"{ncc(WL, Y*FP['L'][k_]):.6e}"])
                print(f"[D2b] {dv}: {len(M[(dv,'H')])} images ({time.time()-t0:.0f}s)")
        finally:
            del vae; gc.collect(); torch.cuda.empty_cache()

    df = rd(p)
    for c_ in ("rho_K","rho_L"): df[c_] = df[c_].astype(float)
    rng = np.random.default_rng(0)
    resid = json.load(open(os.path.join(ROOT,"d2_vae.json")))
    out = {"fingerprints":"raw"}

    for tag, cname in (("K","rho_K"), ("L","rho_L")):
        def contrasts(st):
            per = {}
            for dv in C.DEVICES:
                g   = df[(df.dev==dv) & (df.stage==st)]
                own = g[g.F_dev==dv].sort_values("idx")[cname].to_numpy(float)
                oth = np.mean([g[g.F_dev==o].sort_values("idx")[cname].to_numpy(float)
                               for o in C.DEVICES if o!=dv], axis=0)
                n = min(len(own), np.size(oth))
                if n == 0: HALT(f"empty slice for {dv} at {st}")
                per[dv] = own[:n]-oth[:n]
            return per
        pre, post = contrasts("pre"), contrasts("post")
        Rpre  = float(np.mean([v.mean() for v in pre.values()]))
        Rpost = float(np.mean([v.mean() for v in post.values()]))
        eta   = Rpost/Rpre
        boots = []
        for _ in range(10000):
            a = np.mean([pre[dv][rng.integers(0,len(pre[dv]),len(pre[dv]))].mean()
                         for dv in C.DEVICES])
            b = np.mean([post[dv][rng.integers(0,len(post[dv]),len(post[dv]))].mean()
                         for dv in C.DEVICES])
            boots.append(b/a)
        lo, hi = np.percentile(boots,[2.5,97.5])
        aucs = {st: auc_of(df[(df.stage==st)&(df.dev==df.F_dev)][cname].to_numpy(float),
                           df[(df.stage==st)&(df.dev!=df.F_dev)][cname].to_numpy(float))
                for st in ("pre","post")}
        out[tag] = {"R_pre":Rpre,"R_post":Rpost,"eta":eta,
                    "eta_ci":[float(lo),float(hi)],
                    "auc_pre":aucs["pre"],"auc_post":aucs["post"]}
        r = resid[tag]
        rel = abs(eta - r["eta"])/max(abs(r["eta"]),1e-12)
        overlap = not (hi < r["eta_ci"][0] or lo > r["eta_ci"][1])
        print(f"\n[D2b] {tag}")
        print(f"     RAW         eta {eta:.4f}  CI [{lo:.4f}, {hi:.4f}]  "
              f"AUC {aucs['pre']:.4f} -> {aucs['post']:.4f}")
        print(f"     RESIDUALISED eta {r['eta']:.4f}  CI "
              f"[{r['eta_ci'][0]:.4f}, {r['eta_ci'][1]:.4f}]  "
              f"AUC {r['auc_pre']:.4f} -> {r['auc_post']:.4f}")
        print(f"     difference {rel:.1%} of the residualised value; "
              f"intervals {'OVERLAP' if overlap else 'DO NOT overlap'}")
        out[tag].update({"rel_diff_vs_resid":float(rel), "ci_overlap":bool(overlap)})
    ajson(out, os.path.join(ROOT,"d2_vae_raw.json"), indent=2)

    kr, kd = out["K"], resid["K"]
    print("\n" + "="*72)
    if kr["ci_overlap"] and kr["rel_diff_vs_resid"] < 0.10:
        print(f"CLEAN. eta is insensitive to residualisation ({kr['rel_diff_vs_resid']:.1%}), so")
        print(f"comparing Daxing's {kd['eta']:.4f} to Dresden's 0.3661 is like for like. Report")
        print(f"the residualised value and note this check in the methods.")
    else:
        print(f"NOT INTERCHANGEABLE. eta moves {kr['rel_diff_vs_resid']:.1%} between raw "
              f"({kr['eta']:.4f}) and\nresidualised ({kd['eta']:.4f}). Dresden's 0.3661 was "
              f"measured WITHOUT residualisation, so\nthe raw value is the like-for-like "
              f"comparator. Report both, and say which is which.")
    print(f"\nEither way, both Daxing values exceed Dresden's 0.3661 and its upper limit 0.387:")
    print(f"  raw {kr['eta']:.4f} [{kr['eta_ci'][0]:.4f}, {kr['eta_ci'][1]:.4f}]   "
          f"residualised {kd['eta']:.4f} [{kd['eta_ci'][0]:.4f}, {kd['eta_ci'][1]:.4f}]")
    print("="*72)
d2b()


In [ ]:
# %% 6 · D3 — ten adapters (5 devices x 2 seeds)
#
# COMPLETE CELL. Fixes carried in, each one a failure already met in this notebook:
#   1. SPLIT SIZE FROM THE MANIFEST, not C.N_T. D0 scales the targets to fit scene capacity
#      and allocated T=30 while C.N_T still says 40, so `len(pngs) < C.N_T` could never be
#      satisfied: every run re-wrote the crops and train_meta recorded a count the data did
#      not have. Once D0 has scaled anything, the manifest is the only authority.
#   2. STAMPED CROPS, so training images from an earlier split are discarded on a D0 re-run.
#   3. MANIFEST CACHED. _expected() is called once per adapter per validity check, and each
#      call used to re-read the CSV from Drive.
#   4. Device codes stay strings throughout (MANIF coerces them).
for _n in ("MANIF", "_manifest_stamp"):
    if _n not in globals(): HALT(f"run the D1 cell first — it defines {_n}()")

_MC = {}
def _man():
    k = _manifest_stamp()
    if k not in _MC: _MC.clear(); _MC[k] = MANIF()
    return _MC[k]
def _n_train(): return len(_man()[(C.DEVICES[0], "T")])

def _expected(t):
    _, dev, seed = next(a for a in all_arms() if a[0] == t)
    return {"tag":t, "dev":dev, "seed":seed, "rank":C.RANK, "steps":C.STEPS, "meas":MEAS,
            "n_train":_n_train(), "config_sha":CFG_SHA, "model":C.SD, "dtype":str(DT)}

def avalid(t):
    if not os.path.exists(os.path.join(adir(t), "pytorch_lora_weights.safetensors")): return False
    p = os.path.join(adir(t), "train_meta.json")
    if not os.path.exists(p): return False
    try: m = json.load(open(p))
    except Exception: return False
    return all(m.get(k) == v for k, v in _expected(t).items()) and m.get("effective", False)

def _materialise():
    """Write the T-split crops as PNGs, discarding any that predate the current manifest."""
    M, nT, stamp = _man(), _n_train(), _manifest_stamp()
    sfile = os.path.join(ROOT, "train_png", "_stamp.txt")
    os.makedirs(os.path.join(ROOT, "train_png"), exist_ok=True)
    if os.path.exists(sfile) and open(sfile).read().strip() != stamp:
        for dv in C.DEVICES: shutil.rmtree(tdir(dv), ignore_errors=True)
        print("[D3] manifest changed — discarded the cached training crops")
    made = 0
    for dv in C.DEVICES:
        o = tdir(dv); os.makedirs(o, exist_ok=True)
        have = sorted(glob.glob(os.path.join(o, "*.png")))
        if len(have) != nT:
            for f in have: os.remove(f)
            for i, fp in enumerate(M[(dv, "T")]):
                Image.fromarray(load_rgb(fp).astype(np.uint8)).save(
                    os.path.join(o, f"{i:04d}.png"), compress_level=1)
            made += 1
            print(f"[D3] materialised {dv}: {nT} crops @ {MEAS}")
    open(sfile, "w").write(stamp)
    return nT, made

def d3():
    print("[D3] adapters v3 — manifest-sized, stamped crops, cached manifest\n")
    from diffusers import StableDiffusion3Pipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    nT, _ = _materialise()
    print(f"[D3] {len(all_arms())} adapters | {nT} training images per device "
          f"(manifest; C.N_T={C.N_T} is ignored) | rank {C.RANK} | {C.STEPS} steps\n")

    pp = os.path.join(ROOT, "adapters", "_prompt.pt")
    if os.path.exists(pp):
        z = torch.load(pp, map_location=DEV); pe, po = z["pe"].to(DEV), z["pp"].to(DEV)
    else:
        pi = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT, token=True).to(DEV)
        with torch.no_grad():
            pe, _, po, _ = pi.encode_prompt(prompt=C.CAPTION, prompt_2=C.CAPTION,
                                            prompt_3=C.CAPTION, device=DEV,
                                            num_images_per_prompt=1,
                                            do_classifier_free_guidance=False)
        torch.save({"pe": pe.cpu(), "pp": po.cpu()}, pp); del pi
        gc.collect(); torch.cuda.empty_cache()

    for i, (tag, dv, seed) in enumerate(all_arms(), 1):
        if avalid(tag): print(f"[D3] {i}/{len(all_arms())} skip valid {tag}"); continue
        out = adir(tag)
        if os.path.exists(out): shutil.move(out, out + f"__stale_{int(time.time())}")
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        try:
            pipe = StableDiffusion3Pipeline.from_pretrained(
                C.SD, torch_dtype=DT, token=True, text_encoder=None, text_encoder_2=None,
                text_encoder_3=None, tokenizer=None, tokenizer_2=None,
                tokenizer_3=None).to(DEV)
        except Exception:
            pipe = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT,
                                                            token=True).to(DEV)
            pipe.text_encoder = pipe.text_encoder_2 = pipe.text_encoder_3 = None
        sch, vae, tr = pipe.scheduler, pipe.vae, pipe.transformer
        vae.requires_grad_(False); tr.requires_grad_(False)
        tr.add_adapter(LoraConfig(r=C.RANK, lora_alpha=C.RANK, init_lora_weights="gaussian",
                                  target_modules=list(C.TARGETS)))
        if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
        tr.train()
        ps  = [p for p in tr.parameters() if p.requires_grad]
        if not ps: HALT(f"{tag}: no trainable parameters — LoRA did not attach")
        opt = torch.optim.AdamW(ps, lr=C.LR, weight_decay=1e-4)
        lrs = torch.optim.lr_scheduler.CosineAnnealingLR(opt,
                                                         T_max=math.ceil(C.STEPS/C.GRAD_ACC))
        opt.zero_grad(set_to_none=True)
        files = sorted(glob.glob(os.path.join(tdir(dv), "*.png")))
        if len(files) != nT: HALT(f"{tag}: {len(files)} crops on disk, manifest says {nT}")
        cache = torch.stack([torch.from_numpy(
            np.asarray(Image.open(f).convert("RGB"), np.float32).copy()/127.5 - 1.0
            ).permute(2,0,1) for f in files])
        vsf = float(vae.config.scaling_factor); vsh = float(vae.config.shift_factor or 0.0)
        ts, sg = sch.timesteps.to(DEV), sch.sigmas.to(DEV)
        nts = int(sch.config.num_train_timesteps)
        if len(sg) != nts or len(ts) != nts:
            HALT("scheduler array length mismatch — pin diffusers")
        g = torch.Generator().manual_seed(seed); losses = []; t0 = time.time()
        print(f"\n[D3] {i}/{len(all_arms())} {tag}: device {dv}, seed {seed}")
        for st in range(C.STEPS):
            idx = torch.randint(0, cache.shape[0], (C.BATCH,), generator=g)
            px  = cache[idx].to(DEV, dtype=DT)
            with torch.no_grad():
                lat = vae.encode(px).latent_dist.sample(); lat = ((lat - vsh)*vsf).to(DT)
            noise = torch.randn_like(lat)
            u  = torch.sigmoid(torch.randn(lat.shape[0], device=DEV, dtype=torch.float32))
            ii = (u*nts).long().clamp_(0, nts-1); sig = sg[ii].to(lat.dtype).view(-1,1,1,1)
            pred = tr(hidden_states=((1.0-sig)*lat + sig*noise).to(DT), timestep=ts[ii],
                      encoder_hidden_states=pe.repeat(lat.shape[0],1,1).to(DT),
                      pooled_projections=po.repeat(lat.shape[0],1).to(DT),
                      return_dict=False)[0]
            loss = tF.mse_loss(pred.float(), (noise-lat).float())/C.GRAD_ACC
            loss.backward()
            if (st+1) % C.GRAD_ACC == 0:
                torch.nn.utils.clip_grad_norm_(ps, 1.0); opt.step(); lrs.step()
                opt.zero_grad(set_to_none=True)
            losses.append(float(loss.detach())*C.GRAD_ACC)
            if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
            if (st+1) % 400 == 0:
                el = time.time()-t0
                print(f"    {st+1}/{C.STEPS} loss {np.mean(losses[-400:]):.5f} "
                      f"eta {(el/(st+1))*(C.STEPS-st-1)/60:.1f}min")
        sd = get_peft_model_state_dict(tr)
        bn = float(sum(float(v.float().norm()) for k, v in sd.items() if "lora_B" in k))
        if not (bn > 1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
            HALT(f"{tag}: DEAD adapter (||lora_B||={bn:.3e})")
        os.makedirs(out, exist_ok=True)
        StableDiffusion3Pipeline.save_lora_weights(save_directory=out,
                                                   transformer_lora_layers=sd)
        m = _expected(tag)
        m.update({"loss_tail": float(np.mean(losses[-100:])), "lora_B_norm": bn,
                  "effective": True, "minutes": (time.time()-t0)/60})
        ajson(m, os.path.join(out, "train_meta.json"), indent=2)
        print(f"[D3] saved {tag}: ||lora_B||={bn:.2f} tail={m['loss_tail']:.5f} "
              f"({m['minutes']:.1f} min)")
        del pipe, tr, vae, cache; gc.collect(); torch.cuda.empty_cache()

    bad = [t for t in all_tags() if not avalid(t)]
    if bad: HALT(f"D3 incomplete: {bad}")
    print(f"\n[D3] PASS — {len(all_tags())} adapters\n")
    print(f"{'adapter':14s}{'||lora_B||':>12s}{'tail loss':>12s}{'min':>7s}")
    norms = []
    for t in all_tags():
        m = json.load(open(os.path.join(adir(t), "train_meta.json")))
        norms.append(m["lora_B_norm"])
        print(f"{t:14s}{m['lora_B_norm']:12.2f}{m['loss_tail']:12.5f}{m['minutes']:7.1f}")
    cv = 100*np.std(norms, ddof=1)/np.mean(norms)
    print(f"\n  ||lora_B|| mean {np.mean(norms):.2f}, CV {cv:.1f}%   "
          f"(Dresden SD-3.5 rank-16 ~70, Kodak ~74)")
    if cv > 15:
        print("  *** CV above 15% means the adapters differ substantially in how hard they")
        print("      trained. Report it — D6's per-device theta could reflect that rather")
        print("      than the device.")
if stage("D3"): d3()


In [ ]:
# %% 7 · D4 — generation, paired seed bank
def d4():
    from diffusers import StableDiffusion3Pipeline
    if DEV != "cuda": HALT("D4 needs a GPU")
    bank = [(C.CAPTION, C.GEN_SEED_BASE+j) for j in range(C.G_PER)]
    pipe = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT, token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True); prev = False
    try:
        for tag in all_tags():
            d_ = gdir(tag); os.makedirs(d_, exist_ok=True)
            for t_ in glob.glob(os.path.join(d_,"*.tmp")): os.remove(t_)
            miss = [i for i in range(C.G_PER)
                    if not os.path.exists(os.path.join(d_, f"{i:05d}.png"))]
            if not miss: print(f"[D4] skip {tag}"); continue
            if prev: pipe.unload_lora_weights(); prev = False
            pipe.load_lora_weights(adir(tag)); prev = True
            print(f"[D4] {tag}: {len(miss)} missing"); t0 = time.time()
            for s in range(0, len(miss), C.GEN_BATCH):
                ix = miss[s:s+C.GEN_BATCH]
                gs = [torch.Generator(device=DEV).manual_seed(bank[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs = pipe(prompt=[bank[i][0] for i in ix], num_inference_steps=C.GEN_STEPS,
                                guidance_scale=C.CFG_SCALE, height=MEAS, width=MEAS,
                                generator=gs).images
                for i, im in zip(ix, imgs):
                    t_ = os.path.join(d_, f"{i:05d}.png.tmp")
                    im.save(t_, format="PNG"); os.replace(t_, os.path.join(d_, f"{i:05d}.png"))
                if (s//C.GEN_BATCH) % 25 == 0:
                    el = time.time()-t0
                    print(f"    {s+len(ix)}/{len(miss)}  eta "
                          f"{(el/max(s+len(ix),1))*(len(miss)-s-len(ix))/60:.0f}min")
            print(f"[D4] complete {tag} ({(time.time()-t0)/60:.0f} min)")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[D4] PASS")
if stage("D4"): d4()

In [ ]:
# %% 8 · D5 — measurement, both representations
def d5():
    _s = os.path.join(ROOT,"fingerprints","_manifest_stamp.txt")
    if not os.path.exists(_s) or open(_s).read().strip() != _manifest_stamp():
        HALT("fingerprints were estimated from a different manifest — re-run D1 first")
    KF = torch.stack([torch.from_numpy(np.load(os.path.join(ROOT,"fingerprints",f"K_{d}_E2.npy")))
                      for d in C.DEVICES]).to(_MD)
    LF = torch.stack([torch.from_numpy(np.load(os.path.join(ROOT,"fingerprints",f"L_{d}_E2.npy")))
                      for d in C.DEVICES]).to(_MD)
    p = os.path.join(ROOT,"csv","d5_measure.csv"); done = done_keys(p, ["tag","gen_idx","K"])
    ex = ThreadPoolExecutor(max_workers=4)
    try:
        for tag in all_tags():
            paths = [os.path.join(gdir(tag), f"{i:05d}.png") for i in range(C.G_PER)]
            todo = [i for i in range(C.G_PER)
                    if not all((tag,str(i),d) in done for d in C.DEVICES)]
            if not todo: print(f"[D5] {tag}: complete"); continue
            fut = {i: ex.submit(load_lum, paths[i], False) for i in todo[:6]}; t0 = time.time()
            for pos, i in enumerate(todo):
                Y_np = fut.pop(i).result(); nx = pos+6
                if nx < len(todo): fut[todo[nx]] = ex.submit(load_lum, paths[todo[nx]], False)
                Z = torch.from_numpy(Y_np).to(_MD)
                wk = torch.from_numpy(wres(Y_np)).to(_MD)
                wl = torch.from_numpy(lowmid(Y_np)).to(_MD)
                vk = nccb(wk, Z.unsqueeze(0)*KF).cpu().numpy()
                vl = nccb(wl, Z.unsqueeze(0)*LF).cpu().numpy()
                for j, dv in enumerate(C.DEVICES):
                    if (tag,str(i),dv) in done: continue
                    append_row(p, ["tag","gen_idx","K","rho_K","rho_L"],
                               [tag, i, dv, f"{vk[j]:.6e}", f"{vl[j]:.6e}"])
            print(f"[D5] {tag}: {len(todo)} measured ({time.time()-t0:.0f}s)")
    finally:
        ex.shutdown(wait=True); gc.collect()
        if DEV=="cuda": torch.cuda.empty_cache()
    n = len(done_keys(p, ["tag","gen_idx","K"])); exp = len(all_tags())*C.G_PER*len(C.DEVICES)
    if n != exp: HALT(f"D5 incomplete: {n}/{exp}")
    print(f"[D5] PASS — {n}/{exp} rows")
if stage("D5"): d5()

In [ ]:
# %% 9 · D6 — analysis: ICC, sign-flip, additive decomposition, bounds
def d6():
    rng = np.random.default_rng(0)
    d = pd.read_csv(os.path.join(ROOT,"csv","d5_measure.csv"))
    gates = json.load(open(os.path.join(ROOT,"fingerprints","d1_gates.json")))
    out = {"config_sha":CFG_SHA, "model":C.MODEL_NAME, "devices":list(C.DEVICES), "reps":{}}
    for tag, cname in (("K","rho_K"), ("L","rho_L")):
        R_low = gates[tag]["R_real_lower99"]; R_pt = gates[tag]["R_real"]
        rows = []
        for t, dv, seed in all_arms():
            g = d[d.tag==t]
            own = g[g.K==dv].sort_values("gen_idx")[cname].to_numpy(float)
            oth = [g[g.K==o].sort_values("gen_idx")[cname].to_numpy(float)
                   for o in C.DEVICES if o!=dv]
            n = min([len(own)]+[len(x) for x in oth])
            if n == 0: continue
            rows.append((dv, seed, float((own[:n]-np.mean([x[:n] for x in oth],axis=0)).mean())))
        Rdf = pd.DataFrame(rows, columns=["dev","seed","theta"])
        dm = Rdf.groupby("dev")["theta"].mean(); gm = float(Rdf.theta.mean())
        k, nps = len(dm), Rdf.seed.nunique()
        ms = float(nps*((dm-gm)**2).sum()/(k-1))
        within = float(sum(((Rdf[Rdf.dev==dv].theta-dm[dv])**2).sum() for dv in dm.index)/(k*(nps-1)))
        var_u = max((ms-within)/nps, 0.0); icc = var_u/(var_u+within) if var_u+within>0 else 0.0
        tc = float(sps.t.ppf(1-C.ALPHA/2, df=k-1))
        U = float(gm + tc*dm.std(ddof=1)/np.sqrt(k))
        obs = dm.mean()
        pperm = float(sum(1 for sg in itertools.product([-1,1],repeat=k)
                          if (dm.to_numpy()*np.array(sg)).mean() >= obs) / 2**k)
        agree = sum(1 for dv in dm.index
                    if len(set(np.sign(Rdf[Rdf.dev==dv].theta.to_numpy())))==1)
        print(f"\n{'='*72}\n[{tag}] per-device theta")
        for dv in dm.index:
            v = Rdf[Rdf.dev==dv].sort_values("seed").theta.to_numpy()
            print(f"   {dv}: [{', '.join(f'{x:+.3e}' for x in v)}]  mean {dm[dv]:+.3e}")
        print(f"   grand mean {gm:+.4e} | ICC(1,1) {icc:.3f} | seeds agree {agree}/{k} "
              f"(expected {k/2:.1f})")
        print(f"   exact sign-flip p over {k} device means = {pperm:.4f} (floor {1/2**k:.4f})")
        print(f"   U(99%, n={k}, t={tc:.2f}) {U:+.4e}   lambda_U {U/R_low:.4%} "
              f"(conservative denominator {R_low:.4e})")
        # additive decomposition, cross-seed held out
        M = d.groupby(["tag","K"])[cname].mean().unstack()
        dev_of = {t: t.split("_s")[0] for t in M.index}; seed_of = {t: int(t.split("_s")[1]) for t in M.index}
        byd = {}
        for fs in C.SEEDS:
            tr_ = [t for t in M.index if seed_of[t]==fs]; te = [t for t in M.index if seed_of[t]!=fs]
            if not tr_ or not te: continue
            g0 = M.loc[tr_].values.mean(); b = M.loc[tr_].mean(0) - g0
            for t in te:
                dv = dev_of[t]
                byd.setdefault(dv, []).append(
                    (float(b[dv]-np.mean([b[o] for o in C.DEVICES if o!=dv])),
                     float(M.loc[t,dv]-np.mean([M.loc[t,o] for o in C.DEVICES if o!=dv]))))
        r2cv = rdev = pperm_add = float("nan")
        if byd:
            pr = np.array([v for _,vs in sorted(byd.items()) for v,_ in vs])
            ob = np.array([v for _,vs in sorted(byd.items()) for _,v in vs])
            r2cv = float(1-((ob-pr)**2).sum()/((ob-ob.mean())**2).sum())
            Pd = np.array([np.mean([p for p,_ in v]) for _,v in sorted(byd.items())])
            Od = np.array([np.mean([o for _,o in v]) for _,v in sorted(byd.items())])
            rdev = float(sps.pearsonr(Pd,Od)[0])
            r2f = lambda o,p: 1-((o-p)**2).sum()/((o-o.mean())**2).sum()
            o_ = r2f(Od,Pd)
            nl = [r2f(Od, Pd[list(pm)]) for pm in itertools.permutations(range(len(Pd)))]
            pperm_add = float(sum(1 for v in nl if v>=o_)/len(nl))
            print(f"   additive decomposition: cross-seed R^2 {r2cv:.3f}, device-level r "
                  f"{rdev:+.3f}, exact permutation p {pperm_add:.4f}")
        out["reps"][tag] = {"per_device":dm.to_dict(),"grand_mean":gm,"ICC":icc,
                            "seeds_agree":int(agree),"signflip_p":pperm,"U":U,
                            "lambda_U":U/R_low,"lambda_U_point":U/R_pt,"t_crit":tc,
                            "R_real":R_pt,"R_real_lower99":R_low,
                            "additive_R2_cv":r2cv,"additive_r_device":rdev,
                            "additive_perm_p":pperm_add}
    ajson(out, os.path.join(ROOT,"D6_results.json"), indent=2)
    K = out["reps"]["K"]
    print("\n" + "="*72)
    print("READ THIS THE SAME WAY AS THE KODAK EXPERIMENT")
    print(f"  Mixed signs with a null grand mean and sign-flip p = {K['signflip_p']:.3f} means")
    print("  stable PER-DEVICE BIAS, not leakage: leakage predicts every device favours its own")
    print("  fingerprint, i.e. all theta > 0. A high ICC alone is not evidence of transfer.")
    print(f"  This bound generalises over DEVICES (n={len(C.DEVICES)}), not over adapter seeds.")
    print("  Dresden reference: ICC 0.943, sign-flip p 0.434, lambda_U 1.74% at device level.")
    print("="*72)
if stage("D6"): d6()

## Running it

Set `C.STAGES` and re-run cell 1 each time. Everything is resumable.

| stage | wall | what to check |
|---|---|---|
| `("D0",)` | ~10 min | the per-device counts, one shared resolution, cross-split similarity below 0.95 |
| `("D1",)` | ~2 h | **the PRNU gate**, split-half reliability, the orientation test, LODO κ |
| `("D2",)` | ~30 min | η against Dresden's 0.3661 — does a smartphone ISP pass more or less? |
| `("D3",)` | ~4 h | ten adapters, every ‖lora_B‖ non-zero |
| `("D4",)` | ~5 h | 5000 generations |
| `("D5",)` | ~40 min | 25000 measurement rows |
| `("D6",)` | ~2 min | the verdict |

### The gate is the point
D1 halts if PRNU cannot separate these five bodies on held-out real images. Modern phones
apply multi-frame fusion and heavy denoising, and this is genuinely uncertain in advance.
**If it fails, that is the result** — "PRNU-derived device identity is not recoverable from
this device class at a 1024 px centre crop" is a finding worth reporting, and it would also
explain why the whole question is moot for modern hardware. Do not widen the crop or swap the
extractor to rescue it.

### Reading D6
Identical logic to Kodak. A high ICC with **mixed signs** and a null grand mean is stable
per-device bias, not transfer — leakage predicts every device favours its own fingerprint.
The bound generalises over **devices**, not over adapter seeds, so it is not comparable to
the 0.32 % seed-level headline.

### Switching device groups
One line. OPPO R11: `DEVICES=("3301","3302","3304","3305")`, `MODEL_NAME="OPPO R11"`.
vivo X9: `("4101","4102","4103","4104")`. Both are 4608×3456, so the 1024 crop still fits and
the splits can grow — raise `N_E2` toward 140 to recover Dresden-level fingerprint reliability.